In [0]:
%pip install imbalanced-learn

In [0]:
%restart_python

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp,
    count, avg,
    sum as spark_sum,
    round as spark_round
)
print("✅ Imports ready!")

In [0]:
spark.sql("USE diabetes_medallion")

SILVER_TABLE   = "diabetes_medallion.silver_diabetes"
GOLD_FEATURES  = "diabetes_medallion.gold_features"
GOLD_ANALYTICS = "diabetes_medallion.gold_analytics"

print("=" * 50)
print("      GOLD LAYER CONFIGURATION")
print("=" * 50)
print(f"  Source:      {SILVER_TABLE}")
print(f"  ML Features: {GOLD_FEATURES}")
print(f"  Analytics:   {GOLD_ANALYTICS}")
print("=" * 50)
print("✅ Configuration ready!")

In [0]:
from imblearn.over_sampling import SMOTE
print("✅ SMOTE imported successfully!")

In [0]:
# Read clean data from Silver layer
silver_df = spark.table(SILVER_TABLE)

print("=" * 50)
print("      SILVER DATA LOADED")
print("=" * 50)
print(f"  Total Rows:    {silver_df.count():,}")
print(f"  Total Columns: {len(silver_df.columns)}")

# Show column list
print("\n  Available Columns:")
for i, (col_name, col_type) in enumerate(silver_df.dtypes, 1):
    print(f"    {i:2}. {col_name:25s} → {col_type}")
print("=" * 50)
print("✅ Silver data ready for Gold processing!")

In [0]:
print("=" * 50)
print("      FEATURE SELECTION")
print("=" * 50)

# We select features in 3 groups:
# Group 1 → Target variable (what we predict)
# Group 2 → Numerical features (for ML)
# Group 3 → Categorical features (for analytics)
# We DROP processed_timestamp, is_current 
# → not useful for ML

# Target variable
target_col = "Diabetes_binary"

# Numerical ML features
numerical_features = [
    "BMI",
    "Age",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "Education",
    "Income",
    "Health_Risk_Score"
]

# Binary features
binary_features = [
    "HighBP",
    "HighChol",
    "CholCheck",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "DiffWalk",
    "Sex"
]

# Categorical features (for analytics only)
categorical_features = [
    "BMI_Category",
    "Age_Group"
]

# All ML features combined
ml_features = numerical_features + binary_features

print(f"  Target column:         {target_col}")
print(f"  Numerical features:    {len(numerical_features)}")
print(f"  Binary features:       {len(binary_features)}")
print(f"  Categorical features:  {len(categorical_features)}")
print(f"  Total ML features:     {len(ml_features)}")

# Select only what we need for Gold
gold_df = silver_df.select(
    [target_col] + ml_features + categorical_features
)

print(f"\n  Gold DataFrame shape:")
print(f"  Rows:    {gold_df.count():,}")
print(f"  Columns: {len(gold_df.columns)}")
print("=" * 50)
print("✅ Feature selection complete!")

In [0]:
print("=" * 50)
print("      CORE GOLD TABLE")
print("=" * 50)

# This is the MASTER gold table
# Clean, typed, transformed — real data only
# No SMOTE, no ML specific changes
# This is what ETL/BI consumes downstream

# Add a gold layer timestamp
from pyspark.sql.functions import current_timestamp, lit

gold_core_df = gold_df \
    .withColumn("gold_timestamp", current_timestamp()) \
    .withColumn("data_version",   lit("1.0"))

# Save as core gold Delta table
gold_core_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("diabetes_medallion.gold_diabetes")

# Verify
verify_df = spark.table("diabetes_medallion.gold_diabetes")

print(f"  ✅ Core Gold table saved!")
print(f"  Table:   diabetes_medallion.gold_diabetes")
print(f"  Rows:    {verify_df.count():,}")
print(f"  Columns: {len(verify_df.columns)}")
print(f"\n  Columns:")
for i, (c, t) in enumerate(verify_df.dtypes, 1):
    print(f"    {i:2}. {c:25s} → {t}")
print("=" * 50)
print("✅ Core Gold table ready for ETL/BI!")

In [0]:
from pyspark.sql.functions import (
    count, avg, round as spark_round,
    sum as spark_sum, when, col
)

print("=" * 50)
print("      GOLD ANALYTICS AGGREGATIONS")
print("=" * 50)

# Aggregation 1 — Diabetes rate by BMI Category
print("\n📊 Diabetes Rate by BMI Category:")
bmi_analytics = gold_df.groupBy("BMI_Category") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct"),
        spark_round(avg("BMI"), 2).alias("avg_bmi"),
        spark_round(avg("Health_Risk_Score"), 2).alias("avg_risk_score")
    ) \
    .orderBy("diabetes_rate_pct", ascending=False)
bmi_analytics.show()

# Aggregation 2 — Diabetes rate by Age Group
print("\n📊 Diabetes Rate by Age Group:")
age_analytics = gold_df.groupBy("Age_Group") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct"),
        spark_round(avg("Health_Risk_Score"), 2).alias("avg_risk_score")
    ) \
    .orderBy("diabetes_rate_pct", ascending=False)
age_analytics.show()

# Aggregation 3 — Diabetes rate by Risk Score
print("\n📊 Diabetes Rate by Health Risk Score:")
risk_analytics = gold_df.groupBy("Health_Risk_Score") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct")
    ) \
    .orderBy("Health_Risk_Score")
risk_analytics.show()

# Aggregation 4 — Key health indicators vs diabetes
print("\n📊 Key Health Indicators vs Diabetes:")
gold_df.groupBy("Diabetes_binary") \
    .agg(
        count("*").alias("count"),
        spark_round(avg("BMI"), 2).alias("avg_bmi"),
        spark_round(avg("Age"), 2).alias("avg_age"),
        spark_round(avg("GenHlth"), 2).alias("avg_genhlth"),
        spark_round(avg("Health_Risk_Score"), 2).alias("avg_risk"),
        spark_round(avg("HighBP"), 2).alias("pct_highbp"),
        spark_round(avg("HighChol"), 2).alias("pct_highchol")
    ) \
    .orderBy("Diabetes_binary") \
    .show()

print("=" * 50)
print("✅ Analytics aggregations complete!")

In [0]:
from pyspark.sql.functions import (
    count, avg, round as spark_round,
    sum as spark_sum, col
)

print("=" * 50)
print("      GOLD ANALYTICS AGGREGATIONS")
print("=" * 50)

# We already computed these in Cell 4
# Now we SAVE them as a proper Delta table
# So BI tools can query them directly

# Aggregation 1 — By BMI Category
bmi_analytics = gold_df \
    .groupBy("BMI_Category") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct"),
        spark_round(avg("BMI"), 2).alias("avg_bmi"),
        spark_round(avg("Health_Risk_Score"), 2).alias("avg_risk_score")
    ) \
    .withColumn("analytics_type", lit("BMI_CATEGORY")) \
    .withColumn("analytics_ts",   current_timestamp())

# Aggregation 2 — By Age Group
age_analytics = gold_df \
    .groupBy("Age_Group") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct"),
        spark_round(avg("BMI"), 2).alias("avg_bmi"),
        spark_round(avg("Health_Risk_Score"), 2).alias("avg_risk_score")
    ) \
    .withColumn("analytics_type", lit("AGE_GROUP")) \
    .withColumn("analytics_ts",   current_timestamp())

# Aggregation 3 — By Risk Score
risk_analytics = gold_df \
    .groupBy("Health_Risk_Score") \
    .agg(
        count("*").alias("total_patients"),
        spark_sum("Diabetes_binary").alias("diabetic_count"),
        spark_round(
            spark_sum("Diabetes_binary") * 100.0 / count("*"), 2
        ).alias("diabetes_rate_pct"),
        spark_round(avg("BMI"), 2).alias("avg_bmi"),
        spark_round(avg("Age"), 2).alias("avg_age")
    ) \
    .withColumn("analytics_type", lit("RISK_SCORE")) \
    .withColumn("analytics_ts",   current_timestamp())

print("✅ All 3 aggregations computed!")
print(f"  BMI Category rows:  {bmi_analytics.count()}")
print(f"  Age Group rows:     {age_analytics.count()}")
print(f"  Risk Score rows:    {risk_analytics.count()}")
print("=" * 50)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

print("=" * 50)
print("      SAVING GOLD ANALYTICS TABLE")
print("=" * 50)

# We need to UNION all 3 aggregations
# into ONE analytics table
# But first we need matching columns
# Let's save each as separate partition

# Save BMI analytics
bmi_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("diabetes_medallion.gold_analytics_bmi")

# Save Age analytics
age_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("diabetes_medallion.gold_analytics_age")

# Save Risk Score analytics
risk_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("diabetes_medallion.gold_analytics_risk")

# Verify all 3 saved
print("  Tables saved:")
print(f"  ✅ gold_analytics_bmi   → "
      f"{spark.table('diabetes_medallion.gold_analytics_bmi').count()} rows")
print(f"  ✅ gold_analytics_age   → "
      f"{spark.table('diabetes_medallion.gold_analytics_age').count()} rows")
print(f"  ✅ gold_analytics_risk  → "
      f"{spark.table('diabetes_medallion.gold_analytics_risk').count()} rows")

# Show final BMI analytics as sample
print("\n  Sample — Diabetes Rate by BMI Category:")
spark.table("diabetes_medallion.gold_analytics_bmi") \
     .orderBy("diabetes_rate_pct", ascending=False) \
     .show()

print("=" * 50)
print("✅ Gold analytics tables saved!")

In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

print("=" * 50)
print("      UNIFIED GOLD ANALYTICS TABLE")
print("=" * 50)

# Problem: 3 aggregations have different columns
# Solution: Standardize columns across all 3
# Using a common schema:
# analytics_type, category, total_patients,
# diabetic_count, diabetes_rate_pct,
# avg_bmi, avg_age, avg_risk_score, analytics_ts

# Standardize BMI analytics
bmi_unified = spark.table(
    "diabetes_medallion.gold_analytics_bmi"
) \
.select(
    lit("BMI_CATEGORY").alias("analytics_type"),
    col("BMI_Category").alias("category"),
    col("total_patients"),
    col("diabetic_count"),
    col("diabetes_rate_pct"),
    col("avg_bmi"),
    lit(None).cast("double").alias("avg_age"),
    col("avg_risk_score"),
    current_timestamp().alias("analytics_ts")
)

# Standardize Age analytics
age_unified = spark.table(
    "diabetes_medallion.gold_analytics_age"
) \
.select(
    lit("AGE_GROUP").alias("analytics_type"),
    col("Age_Group").alias("category"),
    col("total_patients"),
    col("diabetic_count"),
    col("diabetes_rate_pct"),
    col("avg_bmi"),
    lit(None).cast("double").alias("avg_age"),
    col("avg_risk_score"),
    current_timestamp().alias("analytics_ts")
)

# Standardize Risk Score analytics
risk_unified = spark.table(
    "diabetes_medallion.gold_analytics_risk"
) \
.select(
    lit("RISK_SCORE").alias("analytics_type"),
    col("Health_Risk_Score").cast("string").alias("category"),
    col("total_patients"),
    col("diabetic_count"),
    col("diabetes_rate_pct"),
    col("avg_bmi"),
    col("avg_age"),
    lit(None).cast("double").alias("avg_risk_score"),
    current_timestamp().alias("analytics_ts")
)

# UNION all 3 into one table
unified_analytics = bmi_unified \
    .union(age_unified) \
    .union(risk_unified)

# Save as single Gold analytics table
unified_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("diabetes_medallion.gold_analytics")

# Verify
verify_df = spark.table("diabetes_medallion.gold_analytics")

print(f"  ✅ Unified analytics table saved!")
print(f"  Table:   diabetes_medallion.gold_analytics")
print(f"  Rows:    {verify_df.count()}")
print(f"  Columns: {len(verify_df.columns)}")

# Show full table
print("\n  Full Analytics Table:")
verify_df.orderBy(
    "analytics_type", "diabetes_rate_pct",
    ascending=[True, False]
).show(20, truncate=False)

# Drop the 3 separate tables — no longer needed
spark.sql("DROP TABLE IF EXISTS diabetes_medallion.gold_analytics_bmi")
spark.sql("DROP TABLE IF EXISTS diabetes_medallion.gold_analytics_age")
spark.sql("DROP TABLE IF EXISTS diabetes_medallion.gold_analytics_risk")
print("\n✅ 3 separate tables dropped — unified table only!")
print("=" * 50)

In [0]:
import pandas as pd
from pyspark.sql.functions import lit, current_timestamp

print("=" * 50)
print("      SMOTE — CLASS BALANCING")
print("=" * 50)

# Step 1 — Convert Spark → Pandas for SMOTE
# SMOTE is a scikit-learn library
# It works on Pandas DataFrames not Spark

# Select only numeric ML features for SMOTE
# Drop categorical columns (BMI_Category, Age_Group)
# SMOTE works on numbers only!

ml_cols = [
    "Diabetes_binary",
    "BMI", "Age", "GenHlth", "MentHlth",
    "PhysHlth", "Education", "Income",
    "Health_Risk_Score", "HighBP", "HighChol",
    "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity",
    "Fruits", "Veggies", "HvyAlcoholConsump",
    "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"
]

# Convert to Pandas
print("  Converting Spark → Pandas...")
pdf = spark.table(SILVER_TABLE) \
           .select(ml_cols) \
           .toPandas()

print(f"  ✅ Pandas DataFrame shape: {pdf.shape}")

# Sample to prevent Community Edition timeout
# Keep ALL minority (diabetic) rows
# Sample majority to 150,000 rows
print("\n  Sampling for SMOTE efficiency...")
pdf_majority = pdf[pdf["Diabetes_binary"] == 0].sample(
    n=150000, random_state=42)
pdf_minority = pdf[pdf["Diabetes_binary"] == 1]
pdf = pd.concat([pdf_majority, pdf_minority]).reset_index(drop=True)
print(f"  Sampled to:  {len(pdf):,} rows")
print(f"  Majority:    {len(pdf_majority):,}")
print(f"  Minority:    {len(pdf_minority):,}")

# Step 2 — Show class distribution BEFORE SMOTE
print("\n  Class Distribution BEFORE SMOTE:")
print(f"  Non-Diabetic (0): "
      f"{(pdf['Diabetes_binary']==0).sum():,} "
      f"({(pdf['Diabetes_binary']==0).mean()*100:.1f}%)")
print(f"  Diabetic     (1): "
      f"{(pdf['Diabetes_binary']==1).sum():,} "
      f"({(pdf['Diabetes_binary']==1).mean()*100:.1f}%)")

# Step 3 — Apply SMOTE
print("\n  Applying SMOTE...")
X = pdf.drop("Diabetes_binary", axis=1)
y = pdf["Diabetes_binary"]

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"  ✅ SMOTE complete!")

# Step 4 — Show class distribution AFTER SMOTE
print("\n  Class Distribution AFTER SMOTE:")
print(f"  Non-Diabetic (0): "
      f"{(y_resampled==0).sum():,} "
      f"({(y_resampled==0).mean()*100:.1f}%)")
print(f"  Diabetic     (1): "
      f"{(y_resampled==1).sum():,} "
      f"({(y_resampled==1).mean()*100:.1f}%)")
print(f"\n  Rows before SMOTE: {len(pdf):,}")
print(f"  Rows after SMOTE:  {len(X_resampled):,}")
print(f"  Synthetic rows:    "
      f"{len(X_resampled) - len(pdf):,}")
print("=" * 50)
print("✅ Class balancing complete!")

In [0]:
import pandas as pd
from pyspark.sql.functions import lit, current_timestamp

print("=" * 50)
print("      SAVING GOLD FEATURES TABLE")
print("=" * 50)

# Step 1 — Combine X and y back together
smote_pdf = X_resampled.copy()
smote_pdf["Diabetes_binary"] = y_resampled.values

# Step 2 — Add a flag to identify synthetic rows
# Original rows:  first 223,199
# Synthetic rows: remaining 156,441
smote_pdf["is_synthetic"] = False
smote_pdf.loc[223199:, "is_synthetic"] = True

print(f"  Original rows:  "
      f"{(~smote_pdf['is_synthetic']).sum():,}")
print(f"  Synthetic rows: "
      f"{smote_pdf['is_synthetic'].sum():,}")
print(f"  Total rows:     {len(smote_pdf):,}")

# Step 3 — Convert Pandas → Spark
print("\n  Converting Pandas → Spark...")
gold_features_df = spark.createDataFrame(smote_pdf) \
    .withColumn("gold_timestamp", current_timestamp()) \
    .withColumn("data_version",   lit("1.0"))

# Step 4 — Save as Gold Features Delta table
gold_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_FEATURES)

# Step 5 — Verify
verify_df = spark.table(GOLD_FEATURES)
print(f"\n  ✅ Gold features table saved!")
print(f"  Table:   {GOLD_FEATURES}")
print(f"  Rows:    {verify_df.count():,}")
print(f"  Columns: {len(verify_df.columns)}")

# Step 6 — Verify class balance
print("\n  Class Balance Verification:")
verify_df.groupBy("Diabetes_binary") \
         .count() \
         .orderBy("Diabetes_binary") \
         .show()

# Step 7 — Verify synthetic vs real split
print("  Synthetic vs Real:")
verify_df.groupBy("is_synthetic") \
         .count() \
         .orderBy("is_synthetic") \
         .show()

print("=" * 50)
print("✅ Gold features table ready for ML!")

In [0]:
import pandas as pd
from pyspark.sql.functions import lit, current_timestamp

print("=" * 50)
print("      SAVING GOLD FEATURES TABLE")
print("=" * 50)

# Step 1 — Combine X and y back together
smote_pdf = X_resampled.copy()
smote_pdf["Diabetes_binary"] = y_resampled.values

# Step 2 — Flag synthetic vs real rows
# Original rows  → first 223,199
# Synthetic rows → remaining 156,441
smote_pdf["is_synthetic"] = False
smote_pdf.loc[223199:, "is_synthetic"] = True

print(f"  Original rows:  "
      f"{(~smote_pdf['is_synthetic']).sum():,}")
print(f"  Synthetic rows: "
      f"{smote_pdf['is_synthetic'].sum():,}")
print(f"  Total rows:     {len(smote_pdf):,}")

# Step 3 — Convert Pandas → Spark
print("\n  Converting Pandas → Spark...")
gold_features_df = spark.createDataFrame(smote_pdf) \
    .withColumn("gold_timestamp", current_timestamp()) \
    .withColumn("data_version",   lit("1.0"))

# Step 4 — Save as Gold Features Delta table
gold_features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_FEATURES)

# Step 5 — Verify
verify_df = spark.table(GOLD_FEATURES)
print(f"\n  ✅ Gold features table saved!")
print(f"  Table:   {GOLD_FEATURES}")
print(f"  Rows:    {verify_df.count():,}")
print(f"  Columns: {len(verify_df.columns)}")

# Step 6 — Verify class balance
print("\n  Class Balance Verification:")
verify_df.groupBy("Diabetes_binary") \
         .count() \
         .orderBy("Diabetes_binary") \
         .show()

# Step 7 — Synthetic vs Real split
print("  Synthetic vs Real:")
verify_df.groupBy("is_synthetic") \
         .count() \
         .orderBy("is_synthetic") \
         .show()

print("=" * 50)
print("✅ Gold features table ready for ML!")

In [0]:
print("=" * 50)
print("      DELTA LAKE OPTIMIZATION")
print("=" * 50)

# Optimize all 3 Gold tables
# ZORDER by most queried columns

# Core Gold table
print("  Optimizing gold_diabetes...")
spark.sql("""
    OPTIMIZE diabetes_medallion.gold_diabetes
    ZORDER BY (Diabetes_binary, BMI, Age)
""")
print("  ✅ gold_diabetes optimized!")

# Analytics table
print("\n  Optimizing gold_analytics...")
spark.sql("""
    OPTIMIZE diabetes_medallion.gold_analytics
    ZORDER BY (analytics_type, diabetes_rate_pct)
""")
print("  ✅ gold_analytics optimized!")

# Features table
print("\n  Optimizing gold_features...")
spark.sql("""
    OPTIMIZE diabetes_medallion.gold_features
    ZORDER BY (Diabetes_binary, BMI, Age)
""")
print("  ✅ gold_features optimized!")

# Show history for all tables
print("\n=== TABLE HISTORIES ===")
for table in [
    "diabetes_medallion.gold_diabetes",
    "diabetes_medallion.gold_analytics",
    "diabetes_medallion.gold_features"
]:
    print(f"\n  {table}:")
    spark.sql(f"DESCRIBE HISTORY {table}") \
         .select("version", "timestamp", "operation") \
         .show(3, truncate=False)

print("=" * 50)
print("✅ All Gold tables optimized!")

In [0]:
print("=" * 60)
print("      FULL PIPELINE RECONCILIATION")
print("      Bronze → Silver → Gold")
print("=" * 60)

# Load all tables
bronze_df  = spark.table("diabetes_medallion.bronze_diabetes")
silver_df  = spark.table("diabetes_medallion.silver_diabetes")
gold_df    = spark.table("diabetes_medallion.gold_diabetes")
feature_df = spark.table("diabetes_medallion.gold_features")

# Row counts
bronze_count  = bronze_df.count()
silver_count  = silver_df.count()
gold_count    = gold_df.count()
feature_count = feature_df.count()

print(f"\n  ROW COUNT FLOW:")
print(f"  {'─' * 45}")
print(f"  {'Bronze (raw):':<30} {bronze_count:>10,}")
print(f"  {'Silver (cleaned):':<30} {silver_count:>10,}")
print(f"  {'  Rows removed:':<30} "
      f"{bronze_count - silver_count:>10,} "
      f"({(bronze_count-silver_count)/bronze_count*100:.1f}%)")
print(f"  {'Gold Core (real data):':<30} {gold_count:>10,}")
print(f"  {'Gold Features (+ SMOTE):':<30} {feature_count:>10,}")
print(f"  {'  Synthetic rows added:':<30} "
      f"{feature_count - gold_count:>10,}")

# Column counts
print(f"\n  COLUMN COUNT FLOW:")
print(f"  {'─' * 45}")
print(f"  {'Bronze columns:':<30} "
      f"{len(bronze_df.columns):>10}")
print(f"  {'Silver columns:':<30} "
      f"{len(silver_df.columns):>10}")
print(f"  {'Gold Core columns:':<30} "
      f"{len(gold_df.columns):>10}")
print(f"  {'Gold Features columns:':<30} "
      f"{len(feature_df.columns):>10}")

# Data integrity checks
print(f"\n  DATA INTEGRITY CHECKS:")
print(f"  {'─' * 45}")

# Check 1 — Silver retention
silver_retention = silver_count / bronze_count * 100
status = "✅ PASS" if silver_retention >= 90 else "❌ FAIL"
print(f"  Silver retention rate:  "
      f"{silver_retention:.2f}%  {status}")

# Check 2 — Gold matches Silver
gold_match = gold_count == silver_count
status = "✅ PASS" if gold_match else "❌ FAIL"
print(f"  Gold = Silver row count: "
      f"{gold_match}  {status}")

# Check 3 — Class balance in features
class_counts = feature_df \
    .groupBy("Diabetes_binary") \
    .count() \
    .collect()
class_0 = [r["count"] for r in class_counts 
           if r["Diabetes_binary"] == 0][0]
class_1 = [r["count"] for r in class_counts 
           if r["Diabetes_binary"] == 1][0]
balanced = class_0 == class_1
status = "✅ PASS" if balanced else "❌ FAIL"
print(f"  SMOTE class balance:     "
      f"{balanced}  {status}")

# Check 4 — No nulls in Gold
gold_nulls = gold_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in gold_df.columns
    if c not in ["gold_timestamp", "data_version"]
]).collect()[0]
total_nulls = sum([gold_nulls[c] for c in gold_nulls.__fields__])
status = "✅ PASS" if total_nulls == 0 else "❌ FAIL"
print(f"  Gold null check:         "
      f"{total_nulls == 0}  {status}")

# Final summary
print(f"\n{'=' * 60}")
print(f"  PIPELINE SUMMARY")
print(f"{'=' * 60}")
print(f"  Source file:     1 CSV file")
print(f"  Bronze rows:     {bronze_count:,}")
print(f"  Silver rows:     {silver_count:,} "
      f"(after quality checks)")
print(f"  Gold core rows:  {gold_count:,} "
      f"(real data for ETL/BI)")
print(f"  Gold ML rows:    {feature_count:,} "
      f"(balanced for ML)")
print(f"  Tables created:  5 Delta tables")
print(f"{'=' * 60}")
print(f"✅ Full pipeline reconciliation PASSED!")
print(f"   Ready for ML Training! 🚀")